# Module 1: Why OpenEnv? — Your First Environments

In this notebook you'll connect to three real hosted OpenEnv environments and interact with each using the same 3-method interface: `reset()`, `step()`, `state()`.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

In [19]:
# !pip install -q -e ./OpenEnv

import os
import sys

repo = os.path.abspath('OpenEnv')
paths = [repo, os.path.join(repo, 'src'), os.path.join(repo, 'envs')]
for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Setup complete!')

Setup complete!


## 1. The Echo Environment

The simplest possible OpenEnv environment — it echoes back whatever you send. Perfect for learning the interface.

Hosted at: `https://openenv-echo-env.hf.space`

In [23]:
from echo_env import EchoEnv

import requests

ECHO_CANDIDATES = [
    'http://localhost:8000',
    'https://openenv-echo-env.hf.space',
]

def pick_working_url(candidates, expected_tokens):
    for base_url in candidates:
        try:
            health = requests.get(f"{base_url}/health", timeout=8)
            if health.status_code != 200:
                continue

            metadata = requests.get(f"{base_url}/metadata", timeout=8).json()
            name = str(metadata.get('name', '')).lower()
            if any(token in name for token in expected_tokens):
                return base_url
        except Exception:
            pass
    return None

ECHO_URL = pick_working_url(ECHO_CANDIDATES, expected_tokens=['echo'])
print(f'Echo URL selected: {ECHO_URL}')

if ECHO_URL is None:
    print('Echo environment is currently unavailable (or wrong service on localhost:8000).')
    print('Run locally in a terminal: python -m echo_env.server.app')
else:
    # EchoEnv extends MCPToolClient — it exposes tools, not raw reset/step actions.
    # MCP methods (list_tools, call_tool) are async; .sync() wraps them automatically
    # via SyncEnvClient.__getattr__, so the same .sync() pattern works here.
    with EchoEnv(base_url=ECHO_URL).sync() as env:
        # reset() starts a new episode
        result = env.reset()
        print('After reset:')
        print(f'  Observation: {result.observation}')
        print(f'  Done: {result.done}')
        print()

        # Discover available tools
        tools = env.list_tools()
        print('Available tools:')
        for tool in tools:
            print(f'  - {tool.name}: {tool.description}')
        print()

        # call_tool() sends a message and returns the result
        response = env.call_tool('echo_message', message='Hello, OpenEnv!')
        print(f'echo_message("Hello, OpenEnv!") -> {response}')

        response = env.call_tool('echo_with_length', message='OpenEnv')
        print(f'echo_with_length("OpenEnv") -> {response}')

        # state() returns episode metadata
        state = env.state()
        print(f'\nState: step_count={state.step_count}')

Echo URL selected: http://localhost:8000
After reset:
  Observation: done=False reward=0.0 metadata={}
  Done: False

Available tools:
  - echo_message: Echo back the provided message.

Args:
    message: The message to echo back

Returns:
    The same message that was provided
  - echo_with_length: Echo back the message with its length.

Args:
    message: The message to echo back

Returns:
    Dictionary with the message and its length

echo_message("Hello, OpenEnv!") -> Hello, OpenEnv!
echo_with_length("OpenEnv") -> {'message': 'OpenEnv', 'length': 7}

State: step_count=3


Three methods. That's the entire API. Every OpenEnv environment works exactly like this.

## 2. OpenSpiel Catch

Now let's connect to a real game. Catch is a simple single-player game from DeepMind's OpenSpiel:

- A ball falls from the top of a 10×5 grid
- You move a paddle left/right to catch it
- Actions: `0` = left, `1` = stay, `2` = right
- Reward: `+1` if caught, `0` if missed

Same 3 methods, completely different game.

In [25]:
from openspiel_env import OpenSpielEnv, OpenSpielAction

import requests

OPENSPIEL_CANDIDATES = [
    'http://localhost:8000',
]

def pick_working_url(candidates, expected_tokens):
    for base_url in candidates:
        try:
            health = requests.get(f"{base_url}/health", timeout=8)
            if health.status_code != 200:
                continue

            metadata = requests.get(f"{base_url}/metadata", timeout=8).json()
            name = str(metadata.get('name', '')).lower()
            if any(token in name for token in expected_tokens):
                return base_url
        except Exception:
            pass
    return None

OPENSPIEL_URL = pick_working_url(OPENSPIEL_CANDIDATES, expected_tokens=['openspiel', 'catch'])
print(f'OpenSpiel URL selected: {OPENSPIEL_URL}')

if OPENSPIEL_URL is None:
    print('OpenSpiel server not detected on http://localhost:8000.')
    print('Run ONE of these (no Docker):')
    print('PowerShell:')
    print('  cd D:/test')
    print('  $env:PYTHONPATH="D:/test/OpenEnv/src;D:/test/OpenEnv/envs"')
    print('  py -m pip install open_spiel')
    print('  py -m openspiel_env.server.app')
    print('WSL:')
    print('  cd /mnt/d/test')
    print('  export PYTHONPATH=/mnt/d/test/OpenEnv/src:/mnt/d/test/OpenEnv/envs')
    print('  python3 -m pip install --user open_spiel')
    print('  python3 -m openspiel_env.server.app')
else:
    with OpenSpielEnv(base_url=OPENSPIEL_URL).sync() as env:
        result = env.reset()
        print('Game: Catch')
        print(f'Legal actions: {result.observation.legal_actions}')
        print(f'Info state length: {len(result.observation.info_state)}')
        print()

        # Play a few steps with a random policy
        import random
        step = 0
        while not result.done:
            legal_actions = result.observation.legal_actions
            if not legal_actions:
                print('No legal actions returned; stopping episode early.')
                break

            action_id = random.choice(legal_actions)
            action_name = {0: 'LEFT', 1: 'STAY', 2: 'RIGHT'}.get(action_id, str(action_id))
            result = env.step(OpenSpielAction(
                action_id=action_id,
                game_name='catch'
            ))
            step += 1
            print(f'Step {step}: {action_name} -> reward={result.reward}, done={result.done}')

        print(f'\nFinal reward: {result.reward}')
        state = env.state()
        print(f'State: step_count={state.step_count}')

OpenSpiel URL selected: http://localhost:8000
Game: Catch
Legal actions: [0, 1, 2]
Info state length: 50

Step 1: STAY -> reward=0.0, done=False
Step 2: STAY -> reward=0.0, done=False
Step 3: STAY -> reward=0.0, done=False
Step 4: RIGHT -> reward=0.0, done=False
Step 5: LEFT -> reward=0.0, done=False
Step 6: STAY -> reward=0.0, done=False
Step 7: RIGHT -> reward=0.0, done=False
Step 8: RIGHT -> reward=0.0, done=False
Step 9: LEFT -> reward=-1.0, done=True

Final reward: -1.0
State: step_count=9


Same pattern: `reset()` → `step()` → check `done`. The observation type is different (`OpenSpielObservation` vs `EchoObservation`), but the interface is identical.

## 3. TextArena Wordle

TextArena is a text-based game environment. Wordle gives you 6 attempts to guess a 5-letter word, with color-coded feedback after each guess.

Hosted at: `https://burtenshaw-textarena.hf.space`

In [26]:
from textarena_env import TextArenaEnv, TextArenaAction

TEXTARENA_URL = 'https://burtenshaw-wordle.hf.space'

with TextArenaEnv(base_url=TEXTARENA_URL).sync() as env:
    result = env.reset()
    print('Wordle prompt:')
    print(result.observation.prompt)
    print()

    # Make a few guesses
    guesses = ['crane', 'slate', 'blind']
    for guess in guesses:
        if result.done:
            break
        result = env.step(TextArenaAction(message=f'[{guess}]'))
        print(f'Guess: {guess}')
        for msg in result.observation.messages:
            print(f'  [{msg.category}] {msg.content}')
        print(f'  Reward: {result.reward}, Done: {result.done}')
        print()

Wordle prompt:
[GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in square brackets (e.g., '[apple]').
Feedback for each letter will be given as follows:
  - G (green): correct letter in the correct position
  - Y (yellow): letter exists in the word but in the wrong position
  - X (wrong): letter is not in the word
Enter your guess to begin.

Guess: crane
  [MESSAGE] 
[GAME] You are Playing Wordle.
A secret 5-letter word has been chosen. You have 6 attempts to guess it.
For each guess, wrap your word in square brackets (e.g., '[apple]').
Feedback for each letter will be given as follows:
  - G (green): correct letter in the correct position
  - Y (yellow): letter exists in the word but in the wrong position
  - X (wrong): letter is not in the word
Enter your guess to begin.

[GAME] [crane]
[GAME] You submitted [crane].
Feedback:
C R A N E
X Y X X X
You have 5 guesses left.
  Reward: 0.0, Done: False

G

## 4. Async vs Sync

OpenEnv clients are async by default. For notebooks and simple scripts, use the `.sync()` wrapper:

```python
# Sync (notebooks, simple scripts)
with EchoEnv(base_url=url).sync() as env:
    result = env.reset()

# Async (production, training loops)
async with EchoEnv(base_url=url) as env:
    result = await env.reset()
```

For this course, we'll use `.sync()` everywhere for simplicity.

## Summary

You connected to three completely different environments — Echo, Catch, Wordle — using the same interface:

| Method | What it does |
|--------|--------------|
| `reset()` | Start a new episode |
| `step(action)` | Take an action, get observation + reward |
| `state()` | Get episode metadata |

The action and observation types change per environment, but the pattern never does.

**Next:** [Module 2](../module-2/README.md) — Using existing environments to build and compare policies.